# MAS Training Notebook - Market Risk Limit Breach Investigation

> Banks deal with huge amounts of money. If traders take on too much risk and markets move against them, the bank can lose millions in a single day. Risk limits are set by the Risk Committee to:

>Protect the bank from catastrophic losses
>Comply with regulatory requirements (Basel III, etc.)
>Ensure no single desk can destabilize the entire firm

> The 3 LLM agents (Planner, Reasoning, Reporting) use hardcoded realistic responses.
> All other agents (SQL, RAG, History, Validator, LangGraph routing) run for real.

### Pipeline Overview
```
User Input
    |
[Planner Agent]              -> Decides execution steps        (MOCK)
    |
[SQL Retrieval Agent]        -> Fetches breach facts (SQLite)  (REAL)
    |
[RAG Retrieval Agent]        -> Fetches policy passages        (REAL)
    |
[Historical Context Agent]   -> Fetches past breach history    (REAL)
    |
[Reasoning Agent]            -> Classifies breach + cites rules (MOCK)
    |
[Validator Agent]            -> Checks citations and schema    (REAL)
    |
[Reporting Agent]            -> Formats final report           (MOCK)
```

---
## Step 1: Set Working Directory and Install Dependencies

In [1]:
import os
import subprocess
import shutil


os.chdir(r"C:\Users\91991\OneDrive\Documents\MAS\mas_project")  # Windows


print("Working directory:", os.getcwd())
print("Files found:", os.listdir("."))

Working directory: C:\Users\91991\OneDrive\Documents\MAS\mas_project
Files found: ['.env', '.ipynb_checkpoints', 'agents', 'data', 'main.py', 'MAS_Project_Training.ipynb', 'MAS_Project_Training_MockMode.ipynb', 'MAS_Project_Training_MockMode_v2.ipynb', 'planner_agent.py', 'README.md', 'reasoning_agent.py', 'reporting_agent.py', 'requirements.txt']


In [2]:
result = subprocess.run(
    ["pip", "install", "langgraph", "langchain-core", "python-dotenv", "-q"],
    capture_output=True, text=True
)
print(result.stdout or "Dependencies installed successfully.")
if result.returncode != 0:
    print("Errors:", result.stderr)

Dependencies installed successfully.


---
## Step 2: Install Mock Agents

This copies 3 pre-written mock agent files into the agents/ folder.
Originals are backed up as *_original.py before being replaced.

In [4]:
AGENTS_DIR = os.path.join(os.getcwd(), "agents")

# Path to the mock files folder - update if you placed it elsewhere
# Download the mock_agents folder from your trainer and point this to it
MOCK_DIR = r"C:\Users\91991\OneDrive\Documents\MAS\mock_agents"  # Windows
# MOCK_DIR = "/Users/yourname/Downloads/mock_agents"             # Mac/Linux

AGENT_FILES = ["planner_agent.py", "reasoning_agent.py", "reporting_agent.py"]

for filename in AGENT_FILES:
    src     = os.path.join(MOCK_DIR, filename)
    dst     = os.path.join(AGENTS_DIR, filename)
    backup  = os.path.join(AGENTS_DIR, filename.replace(".py", "_original.py"))

    if not os.path.exists(src):
        print(f"ERROR: Mock file not found: {src}")
        print("Make sure the mock_agents folder is in the right location.")
        break

    # Backup original only once
    if os.path.exists(dst) and not os.path.exists(backup):
        shutil.copy2(dst, backup)
        print(f"  Backed up: {filename} -> {filename.replace('.py', '_original.py')}")

    shutil.copy2(src, dst)
    print(f"   installed: {filename}")

print("\nAll  agents installed.")

   installed: planner_agent.py
   installed: reasoning_agent.py
   installed: reporting_agent.py

All  agents installed.


---
## Step 3: Set Up the Database

In [5]:
DB_PATH = os.path.join("data", "market_risk.db")

if not os.path.exists(DB_PATH):
    print("Database not found. Creating now...")
    from data.setup_db import setup_database
    setup_database()
else:
    print("Database already exists at:", DB_PATH)

Database already exists at: data\market_risk.db


---
## Step 4: Inspect the Database

In [6]:
import sqlite3

conn = sqlite3.connect(DB_PATH)

print("=" * 65)
print("BREACH SCENARIOS")
print("=" * 65)
rows = conn.execute("""
    SELECT bg.breach_group_id, bg.business_date, bg.threshold_type,
           bg.actual_value, bg.limit_value,
           ROUND((bg.actual_value - bg.limit_value) / bg.limit_value * 100, 1) AS exceedance_pct,
           d.desk_name
    FROM breach_groups bg
    JOIN desk_hierarchy d ON bg.desk_id = d.desk_id
""").fetchall()

for r in rows:
    print(f"\n  Breach ID  : {r[0]}")
    print(f"  Date       : {r[1]}")
    print(f"  Desk       : {r[6]}")
    print(f"  Type       : {r[2]}")
    print(f"  Actual     : {r[3]:,.0f}  |  Limit: {r[4]:,.0f}")
    print(f"  Exceedance : {r[5]}%")

print("\n" + "=" * 65)
print("BREACH HISTORY (last 90 days)")
print("=" * 65)
history = conn.execute("""
    SELECT desk_id, business_date, severity_assigned,
           ROUND((actual_value - limit_value) / limit_value * 100, 1) AS exceedance_pct,
           resolution_days, root_cause_category
    FROM breach_history
    ORDER BY business_date DESC
""").fetchall()
for r in history:
    print(f"  {r[1]} | {r[0]} | {r[2]:8s} | {r[3]}% over | resolved in {r[4]}d | {r[5]}")

conn.close()

BREACH SCENARIOS

  Breach ID  : 557417
  Date       : 2024-02-10
  Desk       : Equity Derivatives Desk
  Type       : VaR
  Actual     : 1,340,000  |  Limit: 1,000,000
  Exceedance : 34.0%

  Breach ID  : 557420
  Date       : 2024-02-11
  Desk       : Fixed Income Trading Desk
  Type       : Stress Loss
  Actual     : 575,000  |  Limit: 500,000
  Exceedance : 15.0%

  Breach ID  : 557431
  Date       : 2024-02-12
  Desk       : Credit Trading Desk
  Type       : Sensitivity
  Actual     : 54,000  |  Limit: 50,000
  Exceedance : 8.0%

BREACH HISTORY (last 90 days)
  2024-01-28 | DESK-EQ-01 | CRITICAL | 28.0% over | resolved in 3d | Delta spike
  2024-01-20 | DESK-FI-01 | HIGH     | 6.0% over | resolved in 3d | Rate shock
  2024-01-15 | DESK-EQ-01 | HIGH     | 21.0% over | resolved in 2d | Vol regime shift
  2024-01-03 | DESK-EQ-01 | HIGH     | 18.0% over | resolved in 2d | Position buildup
  2023-12-18 | DESK-EQ-01 | MEDIUM   | -2.0% over | resolved in 1d | End of day
  2023-12-05 | 

---
## Step 5: Load the Orchestrator

In [8]:
import importlib
import agents.planner_agent
import agents.reasoning_agent
import agents.reporting_agent
importlib.reload(agents.planner_agent)
importlib.reload(agents.reasoning_agent)
importlib.reload(agents.reporting_agent)

from agents.orchestrator import run_investigation
import json

print("Orchestrator loaded. ")

Orchestrator loaded. 


---
## Scenario 1: CRITICAL Breach
Breach 557417 | Equity Derivatives Desk | VaR - 34% over limit | Recurring pattern

In [9]:
print("Running CRITICAL breach investigation...\n")
result_critical = run_investigation(breach_group_id=557417, business_date="2024-02-10")

Running CRITICAL breach investigation...

  [Planner]  Plan: ['sql_retrieve', 'rag_retrieve', 'history', 'reason', 'validate', 'report']  (mock)
  [SQL]      Breach 557417 | Exceedance: 34.0% | Desk: Equity Derivatives Desk
  [RAG]      Retrieved 3 policy passages
  [History]  0 past breaches found (30d: 0, recurring: False)
  [Reasoning] Severity: CRITICAL | Confidence: 95% | Rule: GOV-MR-001:sect-3.1  (mock)
  [Validator] ✅ PASSED
  [Reporting] Report generated  (mock)


In [10]:
print(result_critical.get("report_text", "No report - escalated to human review"))

  MARKET RISK LIMIT BREACH - INVESTIGATION REPORT  [MOCK MODE]
  Breach Group : 557417
  Business Date: 2024-02-10
  Desk         : Equity Derivatives Desk
  Confidence   : 95%
=
  MARKET RISK LIMIT BREACH - INVESTIGATION REPORT  [MOCK MODE]
  Breach Group : 557417
  Business Date: 2024-02-10
  Desk         : Equity Derivatives Desk
  Confidence   : 95%
=
  MARKET RISK LIMIT BREACH - INVESTIGATION REPORT  [MOCK MODE]
  Breach Group : 557417
  Business Date: 2024-02-10
  Desk         : Equity Derivatives Desk
  Confidence   : 95%
=
  MARKET RISK LIMIT BREACH - INVESTIGATION REPORT  [MOCK MODE]
  Breach Group : 557417
  Business Date: 2024-02-10
  Desk         : Equity Derivatives Desk
  Confidence   : 95%
=
  MARKET RISK LIMIT BREACH - INVESTIGATION REPORT  [MOCK MODE]
  Breach Group : 557417
  Business Date: 2024-02-10
  Desk         : Equity Derivatives Desk
  Confidence   : 95%
=
  MARKET RISK LIMIT BREACH - INVESTIGATION REPORT  [MOCK MODE]
  Breach Group : 557417
  Business Date: 2

In [11]:
print("--- AUDIT TRACE ---")
for entry in result_critical.get("trace", []):
    agent = entry.get("agent", "unknown")
    rest  = {k: v for k, v in entry.items() if k != "agent"}
    print(f"  [{agent:25s}] {json.dumps(rest)}")

--- AUDIT TRACE ---
  [planner                  ] {"output": {"plan": ["sql_retrieve", "rag_retrieve", "history", "reason", "validate", "report"]}, "mock": true}
  [sql_retrieval            ] {"rows_returned": 1}
  [rag_retrieval            ] {"docs_returned": 3}
  [history                  ] {"count_90d": 0, "count_30d": 0}
  [reasoning                ] {"attempt": 1, "severity": "CRITICAL", "confidence": 0.95, "mock": true}
  [validator                ] {"passed": true, "issues": []}
  [reporting                ] {"mock": true}


In [12]:
print("--- REASONING OUTPUT ---")
print(json.dumps(result_critical.get("analysis", {}), indent=2))

--- REASONING OUTPUT ---
{
  "severity": "CRITICAL",
  "breach_type": "VaR Limit Breach - Recurring Pattern",
  "rule_applied": "GOV-MR-001:sect-3.1",
  "drivers": [
    "Actual VaR of 1,340,000 exceeds limit of 1,000,000 by 34%",
    "Five breaches in 90 days - systemic recurring pattern",
    "Three breaches within 30-day window triggers mandatory escalation",
    "Root causes include delta spike, vol regime shift, and position buildup"
  ],
  "confidence": 0.95,
  "escalation_required": true,
  "rationale": "A 34% VaR exceedance with 5 recurrences in 90 days constitutes a CRITICAL breach requiring immediate Group Risk Committee escalation."
}


In [13]:
print("--- POLICY PASSAGES RETRIEVED (RAG) ---")
for p in result_critical.get("rules_passages", []):
    print(f"\n  [{p['doc_id']} / {p['snippet_id']}]  score: {p['score']}")
    print(f"  {p['excerpt']}")

--- POLICY PASSAGES RETRIEVED (RAG) ---

  [GOV-MR-001 / sect-3.1]  score: 0.4
  A CRITICAL breach is defined as actual exposure exceeding the approved VaR limit by more than 20%. Immediate escalation to the Group Risk Committee is required within 2 business hours of detection.

  [GOV-MR-002 / sect-1.1]  score: 0.4
  VaR breaches on equity desks must be cross-referenced against stress loss measures before final severity classification. A VaR breach accompanied by stress loss elevation should be upgraded one severity tier.

  [GOV-MR-002 / sect-2.1]  score: 0.2
  Any breach that recurs more than twice within a 30-day window on the same desk and measure must be escalated regardless of individual severity, and a root cause review must be initiated.


---
## Scenario 2: HIGH Breach
Breach 557420 | Fixed Income Trading Desk | Stress Loss - 15% over limit

In [14]:
print("Running HIGH breach investigation...\n")
result_high = run_investigation(breach_group_id=557420, business_date="2024-02-11")

Running HIGH breach investigation...

  [Planner]  Plan: ['sql_retrieve', 'rag_retrieve', 'history', 'reason', 'validate', 'report']  (mock)
  [SQL]      Breach 557420 | Exceedance: 15.0% | Desk: Fixed Income Trading Desk
  [RAG]      Retrieved 4 policy passages
  [History]  0 past breaches found (30d: 0, recurring: False)
  [Reasoning] Severity: HIGH | Confidence: 88% | Rule: GOV-MR-001:sect-3.2  (mock)
  [Validator] ❌ FAILED (1 issues)
    → snippet_id 'sect-3.2' not in retrieved passages
  [Orchestrator] 🔁 Retry 1/2
  [Reasoning] Severity: HIGH | Confidence: 88% | Rule: GOV-MR-001:sect-3.2  (mock)
  [Validator] ❌ FAILED (1 issues)
    → snippet_id 'sect-3.2' not in retrieved passages
  [Orchestrator] ❌ Max retries reached — manual review required


In [15]:
print(result_high.get("report_text", "No report - escalated to human review"))

In [16]:
print("--- AUDIT TRACE ---")
for entry in result_high.get("trace", []):
    agent = entry.get("agent", "unknown")
    rest  = {k: v for k, v in entry.items() if k != "agent"}
    print(f"  [{agent:25s}] {json.dumps(rest)}")

--- AUDIT TRACE ---
  [planner                  ] {"output": {"plan": ["sql_retrieve", "rag_retrieve", "history", "reason", "validate", "report"]}, "mock": true}
  [sql_retrieval            ] {"rows_returned": 1}
  [rag_retrieval            ] {"docs_returned": 4}
  [history                  ] {"count_90d": 0, "count_30d": 0}
  [reasoning                ] {"attempt": 1, "severity": "HIGH", "confidence": 0.88, "mock": true}
  [validator                ] {"passed": false, "issues": ["snippet_id 'sect-3.2' not in retrieved passages"]}
  [reasoning                ] {"attempt": 2, "severity": "HIGH", "confidence": 0.88, "mock": true}
  [validator                ] {"passed": false, "issues": ["snippet_id 'sect-3.2' not in retrieved passages"]}


---
## Scenario 3: MEDIUM Breach
Breach 557431 | Credit Trading Desk | Sensitivity - 8% over limit | First occurrence

In [17]:
print("Running MEDIUM breach investigation...\n")
result_medium = run_investigation(breach_group_id=557431, business_date="2024-02-12")

Running MEDIUM breach investigation...

  [Planner]  Plan: ['sql_retrieve', 'rag_retrieve', 'history', 'reason', 'validate', 'report']  (mock)
  [SQL]      Breach 557431 | Exceedance: 8.0% | Desk: Credit Trading Desk
  [RAG]      Retrieved 4 policy passages
  [History]  0 past breaches found (30d: 0, recurring: False)
  [Reasoning] Severity: MEDIUM | Confidence: 91% | Rule: GOV-MR-001:sect-3.3  (mock)
  [Validator] ❌ FAILED (1 issues)
    → snippet_id 'sect-3.3' not in retrieved passages
  [Orchestrator] 🔁 Retry 1/2
  [Reasoning] Severity: MEDIUM | Confidence: 91% | Rule: GOV-MR-001:sect-3.3  (mock)
  [Validator] ❌ FAILED (1 issues)
    → snippet_id 'sect-3.3' not in retrieved passages
  [Orchestrator] ❌ Max retries reached — manual review required


In [18]:
print(result_medium.get("report_text", "No report - escalated to human review"))

In [19]:
print("--- AUDIT TRACE ---")
for entry in result_medium.get("trace", []):
    agent = entry.get("agent", "unknown")
    rest  = {k: v for k, v in entry.items() if k != "agent"}
    print(f"  [{agent:25s}] {json.dumps(rest)}")

--- AUDIT TRACE ---
  [planner                  ] {"output": {"plan": ["sql_retrieve", "rag_retrieve", "history", "reason", "validate", "report"]}, "mock": true}
  [sql_retrieval            ] {"rows_returned": 1}
  [rag_retrieval            ] {"docs_returned": 4}
  [history                  ] {"count_90d": 0, "count_30d": 0}
  [reasoning                ] {"attempt": 1, "severity": "MEDIUM", "confidence": 0.91, "mock": true}
  [validator                ] {"passed": false, "issues": ["snippet_id 'sect-3.3' not in retrieved passages"]}
  [reasoning                ] {"attempt": 2, "severity": "MEDIUM", "confidence": 0.91, "mock": true}
  [validator                ] {"passed": false, "issues": ["snippet_id 'sect-3.3' not in retrieved passages"]}


---
## Step 6: Compare All 3 Scenarios

In [20]:
results = [
    ("CRITICAL", result_critical),
    ("HIGH",     result_high),
    ("MEDIUM",   result_medium),
]

print(f"{'Scenario':<10} {'Breach ID':<12} {'Desk':<30} {'Exceedance':>12} {'Severity':<10} {'Confidence':>12} {'Validated'}")
print("-" * 100)

for label, r in results:
    facts    = r.get("breach_facts", {})
    analysis = r.get("analysis", {})
    validated = "YES" if r.get("validated") else "NO"
    print(
        f"{label:<10} "
        f"{str(facts.get('breach_group_id', 'N/A')):<12} "
        f"{facts.get('desk_name', 'N/A'):<30} "
        f"{str(facts.get('exceedance_pct', 'N/A')) + '%':>12} "
        f"{analysis.get('severity', 'N/A'):<10} "
        f"{r.get('confidence', 0):.0%}{'':>7} "
        f"{validated}"
    )

Scenario   Breach ID    Desk                             Exceedance Severity     Confidence Validated
----------------------------------------------------------------------------------------------------
CRITICAL   557417       Equity Derivatives Desk               34.0% CRITICAL   95%        YES
HIGH       557420       Fixed Income Trading Desk             15.0% HIGH       88%        NO
MEDIUM     557431       Credit Trading Desk                    8.0% MEDIUM     91%        NO


In [ ]:
# Uncomment and run this cell to restore the original LLM agents

# for filename in ["planner_agent.py", "reasoning_agent.py", "reporting_agent.py"]:
#     original = os.path.join(AGENTS_DIR, filename)
#     backup   = os.path.join(AGENTS_DIR, filename.replace(".py", "_original.py"))
#     if os.path.exists(backup):
#         shutil.copy2(backup, original)
#         print(f"Restored: {filename}")
#     else:
#         print(f"No backup found for {filename}")

print("Restore cell ready - uncomment the lines above and run when needed.")

---
## Summary - Key Concepts

| Concept | Where it appears |
|---|---|
| Multi-agent orchestration | orchestrator.py - LangGraph StateGraph wires all agents |
| Shared state | workflow_state.py - WorkflowState TypedDict passed between all nodes |
| LLM only where needed | Only Planner, Reasoning, Reporting call an LLM |
| Simulated RAG | rag_retrieval_agent.py - keyword scoring mimics vector similarity |
| Hallucination guard | validator_agent.py - rejects any rule citation not in retrieved passages |
| Conditional routing | route_after_validation() - retry, escalate, or report based on state |
| Audit trail | Every agent appends to state trace for full observability |

When you have an API key, run Step 7 to restore the original agents and add your key to .env.